Módulo EBAC - Visualização Avançada (Dash)

In [ ]:

df = pd.read_csv('ecommerce_estatistica.csv')

print(df.head())
print(df.info())

   Unnamed: 0                                             Título  Nota  \
0           1  Kit 10 Cuecas Boxer Lupo Cueca Box Algodão Mas...   4.5   
1           2  Kit Com 10 Cuecas Boxer Algodão Sem Costura Zo...   4.7   
2           3  Kit 10 Cuecas Boxer Mash Algodão Cotton Box Or...   4.6   
3           4  Kit 3 Short Jeans Feminino Cintura Alta Barato...   4.4   
4           5  Blusa + Calça Térmica Treino Futebol Criança I...   4.7   

   N_Avaliações  Desconto            Marca         Material  \
0        3034.0      18.0             lupo          algodão   
1        5682.0      20.0            zorba          algodão   
2        1700.0      22.0             mash          algodão   
3         507.0       9.0     menina linda             jean   
4          58.0       5.0  roupa zero grau  termico unissex   

                Gênero        Temporada  \
0            Masculino   outono/inverno   
1            Masculino     não definido   
2            Masculino  primavera/verão   
3   

Gráficos

In [31]:
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc
from dash.dependencies import Input, Output


df = pd.read_csv('ecommerce_estatistica.csv')

def cria_app(df):
    app = Dash(__name__)


    preco_min = int(df['Preço'].min())
    preco_max = int(df['Preço'].max())

    app.layout = html.Div([
        html.H1('E-Commerce Dashboard Interativo'),

        # PAINEL DE CONTROLES (COMPONENTES INTERATIVOS)
        html.Div([
            html.Div([
                html.Label('Filtrar por Gênero:'),
                dcc.Dropdown(
                    id='filtro-genero',
                    options=[{'label': g, 'value': g} for g in df['Gênero'].dropna().unique()],
                    value=df['Gênero'].dropna().unique()[0],
                    clearable=False
                )
            ], style={'width': '45%', 'display': 'inline-block', 'paddingRight': '20px'}),

            html.Div([
                html.Label('Filtrar Preço Máximo (R$):'),
                dcc.Slider(
                    id='filtro-preco',
                    min=preco_min,
                    max=preco_max,
                    step=10,
                    value=preco_max,
                    marks={i: f'R${i}' for i in range(preco_min, preco_max + 1, int((preco_max - preco_min)/4 or 1))}
                )
            ], style={'width': '45%', 'display': 'inline-block'})
        ], style={'padding': '20px', 'backgroundColor': '#f9f9f9', 'marginBottom': '20px', 'borderRadius': '8px'}),

        html.Br(),

        # GRÁFICOS ATUALIZADOS DINAMICAMENTE
        html.Div([
            html.Div(dcc.Graph(id='grafico-dispersao-dinamico'), style={'width': '50%', 'display': 'inline-block'}),
            html.Div(dcc.Graph(id='grafico-barras-dinamico'), style={'width': '50%', 'display': 'inline-block'}),
        ]),

        html.Br(),
        html.H2("Todos os Gráficos Exibidos:"),


        dcc.Checklist(
            id='checklist-visibilidade',
            options=[
                {'label': ' Histograma ', 'value': 'fig1'},
                {'label': ' Mapa de Calor ', 'value': 'fig3'},
                {'label': ' Pizza ', 'value': 'fig5'},
                {'label': ' Densidade ', 'value': 'fig6'},
                {'label': ' Regressão ', 'value': 'fig7'}
            ],
            value=['fig1', 'fig3', 'fig5', 'fig6', 'fig7'],
            labelStyle={'display': 'inline-block', 'marginRight': '15px'}
        ),

        html.Br(),

        html.Div(dcc.Graph(id='fig1-histograma'), id='div-fig1'),
        html.Div(dcc.Graph(id='fig3-heatmap'), id='div-fig3'),
        html.Div(dcc.Graph(id='fig5-pizza'), id='div-fig5'),
        html.Div(dcc.Graph(id='fig6-violin'), id='div-fig6'),
        html.Div(dcc.Graph(id='fig7-regressao'), id='div-fig7')
    ])


    # CALLBACK 1: Atualiza os dados dos gráficos interativos (Dropdown + Slider)

    @app.callback(
        [Output('grafico-dispersao-dinamico', 'figure'),
         Output('grafico-barras-dinamico', 'figure')],
        [Input('filtro-genero', 'value'),
         Input('filtro-preco', 'value')]
    )
    def atualiza_graficos_interativos(genero_selecionado, preco_maximo):

        df_filtrado = df[(df['Gênero'] == genero_selecionado) & (df['Preço'] <= preco_maximo)]


        fig_scatter = px.scatter(
            df_filtrado, x='Preço', y='Qtd_Vendidos_Cod',
            title=f'Preço vs Qtd Vendida ({genero_selecionado} - Até R${preco_maximo})',
            labels={'Qtd_Vendidos_Cod': 'Qtd Vendida', 'Preço': 'Preço (R$)'}
        )


        df_mat = df_filtrado.groupby('Material', as_index=False)['Preço'].mean()
        fig_bar = px.bar(
            df_mat, x='Material', y='Preço', color='Material',
            title=f'Preço Médio por Material ({genero_selecionado})'
        )
        fig_bar.update_layout(showlegend=False, xaxis_tickangle=-45)

        return fig_scatter, fig_bar


    # CALLBACK 2: Controle de exibição dos gráficos via Checklist

    @app.callback(
        [
            Output('div-fig1', 'style'), Output('div-fig3', 'style'),
            Output('div-fig5', 'style'), Output('div-fig6', 'style'),
            Output('div-fig7', 'style'),
            Output('fig1-histograma', 'figure'), Output('fig3-heatmap', 'figure'),
            Output('fig5-pizza', 'figure'), Output('fig6-violin', 'figure'),
            Output('fig7-regressao', 'figure')
        ],
        Input('checklist-visibilidade', 'value')
    )
    def visibilidade_e_carga_estatica(selected_values):
        graficos = ['fig1', 'fig3', 'fig5', 'fig6', 'fig7']
        estilos = [{} if g in selected_values else {'display': 'none'} for g in graficos]


        fig1 = px.histogram(df, x='Preço', nbins=100, title='Histograma - Distribuição de Preço')

        corr = df[['Preço', 'Desconto_MinMax']].corr()
        fig3 = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Matriz de Correlação')

        contagem_temporada = df['Temporada'].value_counts().reset_index()
        contagem_temporada.columns = ['Temporada', 'Quantidade']
        fig5 = px.pie(contagem_temporada, names='Temporada', values='Quantidade', title='Distribuição por Temporada')

        fig6 = px.violin(df, x='Preço', color='Temporada', orientation='h', points=False, title='Densidade de Preço por Temporada')

        fig7 = px.scatter(df, x='N_Avaliações', y='Preço', trendline='ols', title='Avaliações vs Preço')

        return *estilos, fig1, fig3, fig5, fig6, fig7

    return app

if __name__ == '__main__':
    app = cria_app(df)
    app.run(debug=True, port=8051)

<IPython.core.display.Javascript object>